# 36. Run a stream with the plasticity indicators attached

This notebook learns one eight-task stream and measures the plasticity
indicators while it learns. It is the only way to get those numbers: four of
the five are read at a task boundary, and the paired study keeps two rolling
resume slots rather than one checkpoint per task, so a finished run has no
boundaries left to read.

### What rides along

| Measurement | Where it comes from |
| --- | --- |
| `delta_w_target`, `norm_target`, `angle_target` | the target weights the generator writes, through one fixed reference code |
| `delta_w_generator`, `norm_generator`, `angle_generator` | the generator's own parameters |
| `effective_rank`, `srank` | the classifier's input features, on three fixed image sets |
| `dormant_0.01`, `dormant_0.025`, `dormant_0.1` | ReLU units that never respond, on the same three sets |
| `loss_progress` | how far one update moves the loss, sampled every tenth update |

Each one is written in `uncle/plasticity_indicators.py` and tested there. This
notebook only supplies the images and attaches the observer.

### What it costs

One source job: eight tasks, five epochs each, on the paired study's own
settings. The indicators add one extra forward pass in every ten updates plus a
few forward passes at each task boundary, which is a few percent of the total.
Give the session a real GPU and a budget you are willing to spend. An
interrupted session resumes from the last verified R2 save and the indicator
file picks up at the same place.

### What it does not touch

Nothing here changes `paired_generalization_v2`. This notebook creates its own
study under its own name, so the frozen study and its recorded code revision
stay exactly as they are. Notebooks 19 and 20 are untouched too.

## 1. Load the code

In [ ]:
from pathlib import Path
import sys, os, json, time, shutil, subprocess, importlib.util

IN_COLAB = "google.colab" in sys.modules
if any(name == "uncle" or name.startswith("uncle.") for name in sys.modules):
    raise RuntimeError("Restart the runtime before loading the code.")

# This notebook runs at the current main, not at a pinned revision. The
# indicators were written after the paired study was frozen, so a pinned
# checkout would not contain them.
if IN_COLAB:
    REPO = Path("/content/path-indicators")
    if not REPO.exists():
        subprocess.run(["git", "clone", "https://github.com/sumitasthana/PATH.git", str(REPO)], check=True)
    subprocess.run(["git", "-C", str(REPO), "fetch", "origin"], check=True)
    subprocess.run(["git", "-C", str(REPO), "checkout", "main"], check=True)
    subprocess.run(["git", "-C", str(REPO), "pull", "--ff-only"], check=True)
else:
    REPO = next((p for p in [Path.cwd(), *Path.cwd().parents]
                 if (p / "uncle/task_partition.json").exists()), None)
    if REPO is None:
        raise FileNotFoundError("Open this notebook from a PATH checkout.")

for package in ("boto3", "tqdm", "matplotlib"):
    if importlib.util.find_spec(package) is None:
        if not IN_COLAB:
            raise ImportError("Install the storage and extras dependencies first.")
        subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", package], check=True)
sys.path.insert(0, str(REPO))

from uncle.storage import R2Store
from uncle.paired_study import StudyStore, make_plan, jobs, run_job
from uncle.plasticity_indicators import StreamObserver, fixed_image_sets, indicator_key
from uncle.streams import build_tasks

import torch

HEAD = subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True).strip()
print("Running at:", HEAD)
print("PyTorch:", torch.__version__, "| CUDA build:", torch.version.cuda)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("No GPU. This notebook trains; switch to a GPU runtime before section 7.")

## 2. Settings

In [ ]:
# The study name. This is a new study, separate from paired_generalization_v2,
# so nothing here can disturb the frozen one. Keep the name if you add seeds
# later: the settings are checked against the saved copy and a difference stops
# the notebook rather than silently mixing two protocols.
STUDY_ID = "indicator_stream_v1"
BUCKET = "uncle-experiments"

ORDER = "order_01"            # tasks 3, 0, 9, 5, 17, 1, 7, 14
SEED = 0                      # one seed per session; 0, 1 and 2 are planned

# The held-out Tiny ImageNet task. It is never learned in this order, so its
# images show what the generator does to data it has not been trained on. Task
# 2 is the same probe task notebooks 29 and 35 use, which keeps the numbers
# comparable with the probe results already recorded.
PROBE_TASK = "2"

# Images per fixed set. The sets are fixed because the adapter plasticity study
# first measured rank and dormant units on the current task's images, then had
# to redo the whole analysis: measuring on moving data confuses "the
# representation of this data changed" with "the representation changed".
IMAGES_PER_SET = 256
INCLUDE_EUROSAT = True        # the third set, from a different dataset entirely

# A soft session budget. The run stops at the next verified R2 save once the
# budget is gone, so nothing is lost and the next session resumes there.
MAX_SESSION_MINUTES = 240
SAVE_RESERVE_MINUTES = 15

# A GPU keeps charging until the runtime is given back. This hands it back as
# soon as the run has finished and both files are in R2, whether it finished
# the eighth task or stopped at the budget. It is deliberately not done when
# the cell raises: an unexpected error is worth looking at on a live runtime.
RELEASE_GPU_WHEN_DONE = True

# Task boundaries whose model is kept for probing later. The run writes its
# state into two rolling slots, resume_0 and resume_1, so by the end nothing
# survives from the middle of the stream and a probe can only compare the start
# against the end. Keeping these three lets the same probe be run at two, four
# and six tasks afterwards, against the stream's own state at that point rather
# than a rebuild of it.
#
# The last boundary needs no entry: a completed job writes its own
# checkpoint.pt. Boundary zero needs none either: it is a fresh generator,
# which SEED rebuilds exactly.
#
# Each one is about 0.5 GB in R2 and costs no GPU time and no upload, because
# the file is copied inside R2 from the save the run has just made. Set this to
# () to keep none.
BOUNDARY_CHECKPOINTS = (2, 4, 6)

DATA_ROOT = Path("/content/data" if IN_COLAB else REPO / "data")
EUROSAT_ROOT = Path("/content/eurosat" if IN_COLAB else REPO / "data/eurosat")
if not 0 < SAVE_RESERVE_MINUTES < MAX_SESSION_MINUTES:
    raise ValueError("The reserve must be smaller than the session budget.")
print("Study:", STUDY_ID, "| order:", ORDER, "| seed:", SEED)

## 3. Connect to R2

In [ ]:
def secret(name):
    value = os.environ.get(name)
    if not value and IN_COLAB:
        from google.colab import userdata
        value = userdata.get(name)
    if not value:
        raise RuntimeError("Missing secret: " + name)
    return value

store = R2Store.connect(secret("R2_ENDPOINT"), BUCKET,
    secret("R2_ACCESS_KEY_ID"), secret("R2_SECRET_ACCESS_KEY"))
store.progress = True
CACHE = REPO / "outputs/paired_study" / STUDY_ID
CACHE.mkdir(parents=True, exist_ok=True)

# Where this run's boundary record lives. The JSONL holds one line per task
# boundary. The .baseline.pt beside it holds the two weight vectors from
# boundary zero, which is what relative weight change is measured against: once
# the generator has moved, boundary zero cannot be recreated from anything else
# the run saves, so a resume without it would silently report no movement.
LOG_KEY = indicator_key(STUDY_ID, ORDER, SEED)
BASELINE_KEY = LOG_KEY.replace(".jsonl", ".baseline.pt")
LOG_PATH = CACHE / "indicators" / f"{ORDER}_seed_{SEED}.jsonl"
LOG_PATH.parent.mkdir(parents=True, exist_ok=True)

print("Connected to R2 bucket:", BUCKET)
print("Indicator record:", LOG_KEY)

## 4. Open or create the study

In [ ]:
# A study saved earlier is reopened as it stands. Only a first run writes a new
# plan, and `prepare` refuses to overwrite settings that differ, so two seeds of
# the same study can never be trained under two different configurations.
STUDY_KEY = "uncle/paired_generalization/" + STUDY_ID + "/study.json"
if store.info(STUDY_KEY) is not None:
    plan = json.loads(store.download(STUDY_KEY, CACHE / "cache").read_text(encoding="utf-8"))
    print("Reopened the saved study, recorded at code revision", plan["code_version"][:12])
    if plan["code_version"] != HEAD:
        print("This checkout is at", HEAD[:12],
              "\nThe training settings still match; only the recorded revision differs.")
else:
    plan = make_plan(HEAD, REPO / "uncle/task_partition.json", study_id=STUDY_ID)
    print("Created a new study at code revision", HEAD[:12])

storage = StudyStore(store, plan, CACHE, cleanup_completed=False)
storage.prepare()

# The settings are the paired study's own, so the indicators describe a stream
# trained exactly the way the recorded results were trained.
settings = plan["config"]
print("Backbone:", settings["backbone"], "| epochs per task:", settings["epochs"],
      "| batch:", settings["batch_size"], "| learning rate:", settings["learning_rate"])

ORDER_TASKS = list(plan["orders"][ORDER])
JOB = next(j for j in jobs(plan) if j["id"] == f"sources/{ORDER}/seed_{SEED}")
print("Job:", JOB["id"])


def boundary_key(t):
    """Where a kept boundary model lives, beside the job's own files."""
    return f"{storage.prefix}/{JOB['id']}/boundary_{t}.pt"


# Checked here rather than in section 2, because the number of tasks comes from
# the plan. A boundary outside the stream would be copied from whatever save
# happened to be last, which is worse than refusing.
if any(not isinstance(t, int) or not 0 < t < len(ORDER_TASKS)
       for t in BOUNDARY_CHECKPOINTS):
    raise ValueError(
        f"BOUNDARY_CHECKPOINTS must be whole numbers of tasks between 1 and "
        f"{len(ORDER_TASKS) - 1}. Boundary 0 is a fresh generator that SEED "
        f"rebuilds, and boundary {len(ORDER_TASKS)} is the job's own "
        "checkpoint.pt, so neither is worth keeping.")
print("Tasks, in order:", " ".join(ORDER_TASKS))
if PROBE_TASK in ORDER_TASKS:
    raise ValueError("The probe task is in the learning order, so it is not held out.")
if storage.completed_report(JOB) is not None:
    print("\nThis job is already finished in R2. Section 7 will return its report "
          "without training. If its indicator file is missing, the boundaries are "
          "gone and the job would have to be rerun under a new study name.")

## 5. Build the images

In [ ]:
# One call builds every task this session needs. Indexing the dataset is the
# largest cost before the first optimizer step, so the probe task is built here
# with the training tasks rather than in a second pass. The probe task is never
# trained; it only supplies images.
needed = tuple(ORDER_TASKS + [PROBE_TASK])
tasks = build_tasks(root=DATA_ROOT, partition_path=REPO / "uncle/task_partition.json",
                    partition_seed=42, include=needed, download=True)
print("Loaded Tiny ImageNet images for", len(needed), "tasks")

from torch.utils.data import ConcatDataset

sources = {
    # Held-out images of the tasks this stream learns. These say what happens
    # to the representation of data the model is being trained on.
    "in_stream": ConcatDataset([tasks[name]["test"] for name in ORDER_TASKS]),
    # A Tiny ImageNet task the stream never sees. Same dataset, unseen classes.
    "probe": tasks[PROBE_TASK]["test"],
}

if INCLUDE_EUROSAT:
    from torchvision.datasets import EuroSAT
    from torchvision.transforms import ToTensor
    # Satellite images, already 64x64, which is what the backbone expects. A
    # measure that moves on Tiny ImageNet and stays flat here is saying
    # something different from one that moves on both.
    eurosat = EuroSAT(root=str(EUROSAT_ROOT), transform=ToTensor(), download=True)
    assert eurosat[0][0].shape == (3, 64, 64), f"Unexpected shape {eurosat[0][0].shape}"
    sources["eurosat"] = eurosat

# The same pictures at every boundary, in every run, for every seed and order,
# because the draw is seeded inside the library and not here. Only the model
# differs between two measurements.
image_sets = fixed_image_sets(sources, count=IMAGES_PER_SET)
for name, images in image_sets.items():
    print(f"{name:<10} {tuple(images.shape)}")

## 6. Pick up an interrupted run

In [ ]:
# Both files come back together or not at all. The JSONL alone would raise at
# the first boundary, because weight change has nothing to measure against.
if store.info(LOG_KEY) is not None:
    shutil.copyfile(store.download(LOG_KEY, CACHE / "cache"), LOG_PATH)
    baseline = LOG_PATH.with_suffix(".baseline.pt")
    if store.info(BASELINE_KEY) is None:
        raise RuntimeError(
            f"{LOG_KEY} is in R2 but {BASELINE_KEY} is not. Weight change is "
            "measured against boundary zero, which cannot be recreated. Start a "
            "new study name rather than continuing against a new baseline.")
    shutil.copyfile(store.download(BASELINE_KEY, CACHE / "cache"), baseline)
    written = [json.loads(line) for line in
               LOG_PATH.read_text(encoding="utf-8").splitlines() if line.strip()]
    print(f"Resuming: {len(written)} boundaries already measured, "
          f"up to t = {max(row['t'] for row in written)}.")
else:
    written = []
    LOG_PATH.unlink(missing_ok=True)
    LOG_PATH.with_suffix(".baseline.pt").unlink(missing_ok=True)
    print("Starting fresh: no indicator record in R2 yet.")

# Section 7 counts from this to tell tasks carried over from an earlier session
# apart from tasks it trains itself, which is what makes its time projection
# mean anything.
TASKS_ALREADY_MEASURED = max((row["t"] for row in written), default=0)

# A boundary is measured once and passed. If this session resumes past one that
# was never kept, re-running will not go back for it, so say so now rather than
# leaving a gap to be discovered when the probe is set up.
kept = [t for t in BOUNDARY_CHECKPOINTS if store.info(boundary_key(t)) is not None]
missed = [t for t in BOUNDARY_CHECKPOINTS
          if t not in kept and t <= TASKS_ALREADY_MEASURED]
if BOUNDARY_CHECKPOINTS:
    print("Boundary models already kept:", kept or "none")
if missed:
    print(f"Boundaries {missed} are already past and were not kept. This run "
          "cannot go back for them; only a fresh run under a new study name can.")

## 7. Run the stream with the indicators attached

This cell needs a GPU. It trains eight tasks of five epochs each.

The run saves its state more often than once per task: at every epoch and at
the start of each request. The observer does not need to know which of those is
a task boundary, because boundaries are numbered by how many tasks have been
learned. A number already in the file is returned from before anything is
measured, so epoch saves cost nothing and the first call at each number is
always the save that follows a completed task.

Boundary zero is measured before the first update, on the fresh generator.

### Reading the output while it runs

Every save prints one line, so there is a timestamped heartbeat roughly once an
epoch rather than silence for twenty minutes. The bar counts tasks, and the
line at each task boundary carries the wall clock, how long that task took, how
long the session has been going, and a projection of what is left from the
tasks finished so far. A projection made from one task is a guess; it settles
after two or three.

The step bars underneath come from the trainer and carry tqdm's own elapsed and
remaining time for the task being learned.

In [ ]:
from datetime import datetime
from tqdm.auto import tqdm

TASK_COUNT = len(ORDER_TASKS)


def clock():
    """The runtime's wall clock, so a printed line can be placed in the day."""
    return datetime.now().strftime("%H:%M:%S")


def spell(seconds):
    """A duration as h:mm:ss. These run to hours, so minutes alone would not do."""
    seconds = int(max(seconds, 0))
    return f"{seconds // 3600}:{seconds // 60 % 60:02d}:{seconds % 60:02d}"


# run_job hands every save's pointer here before it calls the observer, so by
# the time a boundary lands this holds the file that save has just written.
last_save = {"pointer": None}
kept_at = []
not_kept_at = []


def remember_save(job, pointer):
    last_save["pointer"] = pointer


def keep_boundary_model(t):
    """Copy the save the run just made to a name that will not be overwritten.

    The copy happens inside R2, from one key to another. Nothing is read down
    to the runtime, nothing is serialised again, and the GPU is not touched.
    Measured against this bucket it takes about twenty seconds for the half a
    gigabyte, against the ten minutes the task itself took.

    A failure here is reported and the run carries on. The indicator record is
    already uploaded by this point, and losing an hour of GPU time over a
    storage hiccup would cost more than the model this keeps.
    """
    pointer = last_save["pointer"]
    if pointer is None:
        print(f"\n    No save pointer at boundary {t}; model not kept.")
        not_kept_at.append(t)
        return
    if pointer["active"] is not None:
        # Numbering makes this impossible: the first save at a boundary number
        # is the one after the request finished, where no request is active.
        # Checked anyway, because copying a mid-task save under a boundary name
        # would hand the probe a model from the wrong place in the stream.
        print(f"\n    The save at boundary {t} is mid-request; model not kept.")
        not_kept_at.append(t)
        return
    source = pointer["checkpoint"]
    destination = boundary_key(t)
    try:
        store.client.copy_object(
            Bucket=store.bucket, Key=destination,
            CopySource={"Bucket": store.bucket, "Key": source["key"]},
            MetadataDirective="COPY")
        # The sha256 travels in the object's metadata, so the copy can be
        # checked against the original without downloading either one.
        there = store.info(destination)
        if (there is None or there["ContentLength"] != source["bytes"]
                or there.get("Metadata", {}).get("sha256") != source["sha256"]):
            raise RuntimeError("the copy does not match the original")
    except Exception as failure:
        print(f"\n    Could not keep the boundary {t} model: {failure}")
        not_kept_at.append(t)
        return
    kept_at.append(t)


class UploadingObserver(StreamObserver):
    """The observer, plus an upload and a timed line whenever a save happens.

    The Colab disk disappears when the session ends, so a record that only
    lives there is lost with it. Uploading at each boundary means an
    interrupted session keeps every task it finished.

    Every save prints something. A save that lands a new boundary prints the
    task's timing and its two headline measurements. A save at an epoch inside
    a task measures nothing, because that boundary number is already written,
    and prints a one-line heartbeat instead. That is what keeps the output
    moving while a single task trains for twenty minutes.
    """

    def __init__(self, *arguments, bar, already_done, **named):
        super().__init__(*arguments, **named)
        self.bar = bar
        self.started_from = already_done   # tasks finished before this session
        self.began = time.monotonic()
        self.last_boundary = self.began

    def boundary(self, trainer, t, task):
        row = super().boundary(trainer, t, task)
        now = time.monotonic()

        if row is None:
            # An epoch save, or the save at the start of a request. Nothing was
            # measured, so this is a heartbeat and nothing more. The last save
            # of the job lands after the final task, where there is no next one
            # to name.
            where = (f"still on task {t + 1} of {TASK_COUNT}" if t < TASK_COUNT
                     else "all tasks measured, closing the job")
            self.bar.set_postfix_str(f"{spell(now - self.began)} in")
            self.bar.write(f"[{clock()}] saved, {where}"
                           f"   session {spell(now - self.began)}")
            return None

        store.upload(self.log.baseline_path, BASELINE_KEY, overwrite=True)
        store.upload(self.log.path, LOG_KEY, overwrite=True)
        # After the record is safely up, never before: if keeping the model
        # goes wrong, the measurements of this boundary are already in R2.
        if t in BOUNDARY_CHECKPOINTS:
            keep_boundary_model(t)
        self.bar.n = t
        self.bar.refresh()

        if t == 0:
            self.bar.write(f"[{clock()}] boundary 0 measured on the fresh generator, "
                           "before any training")
        else:
            # The projection uses only the tasks this session trained. Tasks
            # carried over from an earlier session cost this session nothing,
            # so counting them would make every task look instant. The floor of
            # one covers a record with a gap in it, where this count could come
            # out at zero and the line would be a crash instead of a guess.
            here = max(t - self.started_from, 1)
            each = (now - self.began) / here
            left = (TASK_COUNT - t) * each
            self.bar.write(
                f"[{clock()}] task {task} done, boundary {t} of {TASK_COUNT}"
                f"   this task {spell(now - self.last_boundary)}"
                f"   session {spell(now - self.began)}"
                f"   about {spell(left)} left"
                f"\n         moved {row['delta_w_generator']:.4f}, "
                f"turned {row['angle_generator']:.3f} degrees, "
                f"rank {row['effective_rank_in_stream']:.2f}"
                + (f"\n         model kept for probing at boundary {t}"
                   if t in kept_at else ""))
        self.last_boundary = now
        return row


if not torch.cuda.is_available():
    raise RuntimeError("This cell trains. Switch to a GPU runtime and run all cells again.")

bar = tqdm(total=TASK_COUNT, initial=TASKS_ALREADY_MEASURED,
           desc="Tasks measured", unit="task")
observer = UploadingObserver(LOG_PATH, image_sets, bar=bar,
                             already_done=TASKS_ALREADY_MEASURED,
                             study=STUDY_ID, order=ORDER, seed=SEED)

deadline = time.monotonic() + (MAX_SESSION_MINUTES - SAVE_RESERVE_MINUTES) * 60
store.progress = False     # one bar per transfer would bury the training bars
began = time.monotonic()
finished = False
print(f"[{clock()}] starting on {torch.cuda.get_device_name(0)}, "
      f"budget {MAX_SESSION_MINUTES} minutes including a {SAVE_RESERVE_MINUTES} minute reserve")
try:
    report = run_job(plan, JOB, tasks, storage, device="cuda", progress=True,
                     stop_requested=lambda: time.monotonic() >= deadline,
                     on_boundary=remember_save, observer=observer)
    finished = True
    print(f"\n[{clock()}] status: {report['status']}, "
          f"after {spell(time.monotonic() - began)}")
    if report["status"] == "complete":
        print("Final accuracies:", report["final_accuracies"])
    else:
        print("Budget reached at a verified save. Run this notebook again with the "
              "same settings to continue from here.")
finally:
    bar.close()
    store.progress = True
    rows = [json.loads(line) for line in
            LOG_PATH.read_text(encoding="utf-8").splitlines() if line.strip()] \
        if LOG_PATH.exists() else []
    print(f"Indicator record: {len(rows)} boundaries in {LOG_KEY}")
    if BOUNDARY_CHECKPOINTS:
        print(f"Boundary models kept this session: {kept_at or 'none'}"
              + (f", failed at {not_kept_at}" if not_kept_at else ""))
    del tasks
    import gc
    gc.collect()
    torch.cuda.empty_cache()

# Outside the finally on purpose. A run that raised is left on a live runtime,
# because an unexpected error is worth looking at with the session still there.
if finished and RELEASE_GPU_WHEN_DONE and IN_COLAB:
    from google.colab import runtime
    print(f"[{clock()}] both files are in R2. Releasing the Colab runtime.")
    runtime.unassign()
elif finished and not RELEASE_GPU_WHEN_DONE:
    print(f"[{clock()}] the GPU is still assigned and still charging. "
          "Release it yourself, or set RELEASE_GPU_WHEN_DONE in section 2.")

## 8. What comes next

A finished run leaves nine lines in `uncle/indicators/`: boundary zero plus one
per task. Repeat this notebook with `SEED = 1` and `SEED = 2` to get a spread,
then open notebook 37 on a CPU runtime to read the files, build the table and
draw the curves. Notebook 37 downloads records, never models, so it needs no
GPU.

It also leaves a model at each boundary in `BOUNDARY_CHECKPOINTS`, saved beside
the job as `boundary_2.pt`, `boundary_4.pt` and `boundary_6.pt`. Those are the
stream's own state after that many tasks, in the same shape as the job's
`checkpoint.pt`, so the probe can read them the way it already reads a finished
source model. Without them the probe can only compare a fresh generator against
the finished one, because the run's two rolling slots keep nothing from the
middle.

A job that is already finished in R2 returns its report without training, so it
keeps nothing. Seed 0 finished before this section existed and has no boundary
models; only a fresh run under a new study name would give it any.